In [1]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/seoultechpse/fenicsx-colab.git"
ROOT = Path("/content")
REPO_DIR = ROOT / "fenicsx-colab"

subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

USE_COMPLEX = False  # <--- Set True ONLY if you need complex PETSc
USE_CLEAN = False    # <--- Set True to remove existing environment

opts_str = " ".join(
  [o for c, o in [(USE_COMPLEX, "--complex"), (USE_CLEAN, "--clean")] if c]
)

get_ipython().run_line_magic("run", f"{REPO_DIR / 'setup_fenicsx.py'} {opts_str}")

🔧 FEniCSx Setup Configuration
PETSc type      : real
Clean install   : False

⚠️  Google Drive not mounted — using local cache (/content)

🔧 Installing FEniCSx environment...

🔍 Verifying PETSc type...
✅ Installed: Real PETSc (float64)

✨ Loading FEniCSx Jupyter magic... %%fenicsx registered

✅ FEniCSx setup complete!

Next steps:
  1. Run %%fenicsx --info to verify installation
  2. Use %%fenicsx in cells to run FEniCSx code
  3. Use -np N for parallel execution (e.g., %%fenicsx -np 4)

📌 Note: Real PETSc is installed
   - Recommended for most FEM problems
   - For complex problems, reinstall with --complex


---

In [2]:
%%fenicsx -np 4 --time

#!/usr/bin/env python3
"""
Parallel Mesh Reading and Partitioning in FEniCSx

Run with: mpirun -n 4 python3 parallel_mesh_demo.py

This script demonstrates:
- Parallel mesh creation and partitioning
- Ghost cell modes (none vs shared_facet)
- Cell ownership and index mapping
- Connectivity in parallel
- Communication patterns
"""

from mpi4py import MPI
import numpy as np
import sys

import dolfinx
import dolfinx.mesh

# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def print_rank0(comm, message):
    """Print only from rank 0"""
    if comm.rank == 0:
        print(message, flush=True)

def print_all_ranks(comm, message, label=""):
    """Print from all ranks with synchronization"""
    for rank in range(comm.size):
        if comm.rank == rank:
            if label:
                print(f"[Rank {rank}] {label}: {message}", flush=True)
            else:
                print(f"[Rank {rank}] {message}", flush=True)
        comm.Barrier()

def inspect_mesh(comm, mesh, ghost_mode_name):
    """Inspect mesh properties on all processes"""
    topology = mesh.topology
    tdim = topology.dim
    cell_map = topology.index_map(tdim)

    # Gather statistics
    local_cells = cell_map.size_local
    ghost_cells = cell_map.num_ghosts
    local_range = cell_map.local_range

    # Print from each rank
    print_all_ranks(
        comm,
        f"Local cells: {local_cells}, Ghost cells: {ghost_cells}, "
        f"Range: [{local_range[0]}, {local_range[1]})",
        label=f"{ghost_mode_name}"
    )

    return local_cells, ghost_cells

# ============================================================================
# MAIN PARALLEL DEMO
# ============================================================================

def main():
    comm = MPI.COMM_WORLD
    rank = comm.rank
    size = comm.size

    # Mesh resolution (change this to test different sizes)
    mesh_nx = 100  # Number of cells in x direction
    mesh_ny = 100  # Number of cells in y direction

    print_rank0(comm, "="*70)
    print_rank0(comm, "PARALLEL MESH READING DEMO")
    print_rank0(comm, "="*70)
    print_rank0(comm, f"\nRunning on {size} processes\n")
    print_rank0(comm, f"Mesh resolution: {mesh_nx}×{mesh_ny} ({mesh_nx*mesh_ny*2} cells)\n")

    # ========================================================================
    # 1. CREATE MESH WITHOUT GHOST CELLS
    # ========================================================================
    print_rank0(comm, "\n1. Creating mesh with GhostMode.none...")

    mesh_no_ghost = dolfinx.mesh.create_unit_square(
        comm, mesh_nx, mesh_ny,
        ghost_mode=dolfinx.mesh.GhostMode.none
    )

    local_cells, ghost_cells = inspect_mesh(comm, mesh_no_ghost, "No Ghost")

    # Compute load balance
    min_cells = comm.allreduce(local_cells, op=MPI.MIN)
    max_cells = comm.allreduce(local_cells, op=MPI.MAX)
    avg_cells = comm.allreduce(local_cells, op=MPI.SUM) / size

    print_rank0(comm, f"\n  Load balance:")
    print_rank0(comm, f"    Min cells: {min_cells}")
    print_rank0(comm, f"    Max cells: {max_cells}")
    print_rank0(comm, f"    Avg cells: {avg_cells:.1f}")
    print_rank0(comm, f"    Balance ratio: {max_cells/min_cells:.2f}")

    # ========================================================================
    # 2. CREATE MESH WITH GHOST CELLS
    # ========================================================================
    print_rank0(comm, "\n2. Creating mesh with GhostMode.shared_facet...")

    mesh_with_ghost = dolfinx.mesh.create_unit_square(
        comm, mesh_nx, mesh_ny,
        ghost_mode=dolfinx.mesh.GhostMode.shared_facet
    )

    local_cells, ghost_cells = inspect_mesh(comm, mesh_with_ghost, "With Ghost")

    # ========================================================================
    # 3. EXAMINE CONNECTIVITY
    # ========================================================================
    print_rank0(comm, "\n3. Examining cell-to-vertex connectivity...")

    topology = mesh_with_ghost.topology
    tdim = topology.dim
    topology.create_connectivity(tdim, 0)
    cell_to_vertices = topology.connectivity(tdim, 0)

    # Show first few cells from each rank
    num_show = min(3, cell_to_vertices.num_nodes)

    for rank_i in range(size):
        if rank == rank_i:
            print(f"\n[Rank {rank}] First {num_show} cells:", flush=True)
            for i in range(num_show):
                vertices = cell_to_vertices.links(i)
                print(f"  Cell {i}: vertices {vertices}", flush=True)
        comm.Barrier()

    # ========================================================================
    # 4. LOCAL TO GLOBAL INDEX MAPPING
    # ========================================================================
    print_rank0(comm, "\n4. Local to global index mapping...")

    cell_map = topology.index_map(tdim)

    # Show mapping for first few cells
    num_cells = cell_map.size_local
    indices_to_show = np.array(list(range(min(3, num_cells))), dtype=np.int32)

    for rank_i in range(size):
        if rank == rank_i:
            print(f"\n[Rank {rank}] Index mapping:", flush=True)
            for local_idx in indices_to_show:
                global_idx = cell_map.local_to_global(np.array([local_idx], dtype=np.int32))[0]
                print(f"  Local {local_idx} → Global {global_idx}", flush=True)
        comm.Barrier()

    # ========================================================================
    # 5. GHOST CELL INFORMATION
    # ========================================================================
    print_rank0(comm, "\n5. Ghost cell information...")

    num_ghost = cell_map.num_ghosts
    total_local = num_cells + num_ghost

    print_all_ranks(
        comm,
        f"Owned: {num_cells}, Ghost: {num_ghost}, Total accessible: {total_local}",
        label="Ghost info"
    )

    # Show ghost cell indices
    if num_ghost > 0:
        for rank_i in range(size):
            if rank == rank_i and num_ghost > 0:
                print(f"\n[Rank {rank}] Ghost cells (local indices):", flush=True)
                ghost_start = num_cells
                ghost_end = num_cells + min(5, num_ghost)
                for i in range(ghost_start, ghost_end):
                    vertices = cell_to_vertices.links(i)
                    global_idx = cell_map.local_to_global(np.array([i], dtype=np.int32))[0]
                    print(f"  Local {i} (Global {global_idx}): vertices {vertices}",
                          flush=True)
            comm.Barrier()

    # ========================================================================
    # 6. VERTEX OWNERSHIP AND SHARING
    # ========================================================================
    print_rank0(comm, "\n6. Vertex ownership...")

    vertex_map = topology.index_map(0)
    local_vertices = vertex_map.size_local
    ghost_vertices = vertex_map.num_ghosts

    print_all_ranks(
        comm,
        f"Owned vertices: {local_vertices}, Ghost vertices: {ghost_vertices}",
        label="Vertices"
    )

    # ========================================================================
    # 7. MESH QUALITY AND STATISTICS
    # ========================================================================
    print_rank0(comm, "\n7. Gathering mesh statistics...")

    # Global totals
    total_cells = cell_map.size_global
    total_vertices = vertex_map.size_global

    print_rank0(comm, f"\n  Global mesh:")
    print_rank0(comm, f"    Total cells: {total_cells}")
    print_rank0(comm, f"    Total vertices: {total_vertices}")

    # Per-process memory (approximate)
    cells_per_proc = num_cells + num_ghost
    vertices_per_proc = local_vertices + ghost_vertices

    print_all_ranks(
        comm,
        f"Cells in memory: {cells_per_proc}, Vertices in memory: {vertices_per_proc}",
        label="Memory"
    )

    # ========================================================================
    # 8. COMMUNICATION PATTERN SIMULATION
    # ========================================================================
    print_rank0(comm, "\n8. Simulating communication patterns...")

    # Create a simple function to test scatter operations
    V = dolfinx.fem.functionspace(mesh_with_ghost, ("Lagrange", 1))
    u = dolfinx.fem.Function(V)

    # Set values on owned DOFs
    u.x.array[:] = rank  # Each process sets its rank as value

    print_rank0(comm, "\n  Before scatter_forward:")
    for rank_i in range(size):
        if rank == rank_i:
            print(f"[Rank {rank}] DOF values (first 5): {u.x.array[:5]}", flush=True)
        comm.Barrier()

    # Scatter forward: owner → ghost
    u.x.scatter_forward()

    print_rank0(comm, "\n  After scatter_forward:")
    for rank_i in range(size):
        if rank == rank_i:
            print(f"[Rank {rank}] DOF values (first 5): {u.x.array[:5]}", flush=True)
        comm.Barrier()

    # ========================================================================
    # 9. SUMMARY
    # ========================================================================
    print_rank0(comm, "\n" + "="*70)
    print_rank0(comm, "SUMMARY")
    print_rank0(comm, "="*70)

    print_rank0(comm, f"\n✓ Parallel execution:")
    print_rank0(comm, f"  - Processes: {size}")
    print_rank0(comm, f"  - Cells per process: {min_cells}-{max_cells}")
    print_rank0(comm, f"  - Load balance: {max_cells/min_cells:.2f}")

    print_rank0(comm, f"\n✓ Ghost cells (shared_facet mode):")
    total_ghost = comm.allreduce(ghost_cells, op=MPI.SUM)
    print_rank0(comm, f"  - Total ghost cells across all processes: {total_ghost}")
    print_rank0(comm, f"  - Average ghost cells per process: {total_ghost/size:.1f}")

    print_rank0(comm, f"\n✓ Memory overhead:")
    overhead = (total_ghost / total_cells) * 100
    print_rank0(comm, f"  - Ghost cell overhead: {overhead:.1f}%")

    print_rank0(comm, "\n" + "="*70)
    print_rank0(comm, "Parallel mesh demo completed successfully!")
    print_rank0(comm, "="*70)

# ============================================================================
# ENTRY POINT
# ============================================================================

if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        rank = MPI.COMM_WORLD.rank
        print(f"[Rank {rank}] ERROR: {e}", file=sys.stderr, flush=True)
        MPI.COMM_WORLD.Abort(1)

PARALLEL MESH READING DEMO

Running on 4 processes

Mesh resolution: 100×100 (20000 cells)


1. Creating mesh with GhostMode.none...
[Rank 0] No Ghost: Local cells: 5026, Ghost cells: 0, Range: [0, 5026)
[Rank 1] No Ghost: Local cells: 4993, Ghost cells: 0, Range: [5026, 10019)
[Rank 2] No Ghost: Local cells: 5013, Ghost cells: 0, Range: [10019, 15032)
[Rank 3] No Ghost: Local cells: 4968, Ghost cells: 0, Range: [15032, 20000)

  Load balance:
    Min cells: 4968
    Max cells: 5026
    Avg cells: 5000.0
    Balance ratio: 1.01

2. Creating mesh with GhostMode.shared_facet...
[Rank 0] With Ghost: Local cells: 5026, Ghost cells: 186, Range: [0, 5026)
[Rank 1] With Ghost: Local cells: 4993, Ghost cells: 76, Range: [5026, 10019)
[Rank 2] With Ghost: Local cells: 5013, Ghost cells: 189, Range: [10019, 15032)
[Rank 3] With Ghost: Local cells: 4968, Ghost cells: 79, Range: [15032, 20000)

3. Examining cell-to-vertex connectivity...

[Rank 0] First 3 cells:
  Cell 0: vertices [2588    0 2589]